# BYD BRASIL — DIAGNÓSTICO COMPLETO
## Analise Descritiva: A Historia por Tras dos Numeros

**Autor:** Matheus Mendes
**Data:** 19/julho/2026
**Objetivo:** Narrativa visual completa — 6 capitulos + 1 sintese

---

### Como ler este notebook

Cada capitulo e **narrativa + visualizacao + interpretacao**. Execute celula por celula.
Dados sao reais (BCB SGS API) e simulacoes sao Monte Carlo com 10.000 caminhos.

**Convencoes de cor:**
- Vermelho = risco / alerta / BYD share caindo
- Verde = oportunidade / BYD share subindo
- Amarelo = zona de atencao / volatilidade
- Azul = contexto neutro / mercado geral


In [ ]:
import json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyBboxPatch
import seaborn as sns
from pathlib import Path

%matplotlib inline
warnings.filterwarnings('ignore')

OUT = Path(r'C:\Users\mathe\code_space\orchestration\value-factory\case-studies\byd-camacari-2025-2027\d2-econometric-vulnerability\outputs')
with open(OUT / 'computed_data.json') as f:
    D = json.load(f)

plt.rcParams.update({
    'figure.facecolor':'#0d1117','axes.facecolor':'#161b22',
    'axes.edgecolor':'#30363d','axes.labelcolor':'#c9d1d9',
    'xtick.color':'#8b949e','ytick.color':'#8b949e',
    'text.color':'#c9d1d9','grid.color':'#21262d','grid.linewidth':0.8,
    'axes.titlesize':13,'axes.labelsize':10,
    'font.family':'DejaVu Sans','axes.spines.top':False,'axes.spines.right':False,
})

C_RED='#f85149'; C_GREEN='#3fb950'; C_YELLOW='#d29922'; C_BLUE='#58a6ff'
C_PURPLE='#bc8cff'; C_ORANGE='#ffa657'; C_GRAY='#8b949e'

print(f"PTAX atual: BRL {D['ptax']['atual']}  |  Vol 30d: {D['ptax']['vol30_annualized']}% a.a.")
print(f"Composite: {D['composite']['composite_pct']}/100  |  Supply: {D['composite']['scores']['supply']}/100")
plt.rcParams["text.usetex"] = False


<frozen importlib._bootstrap>:491: Warning: Numpy built with MINGW-W64 on Windows 64 bits is experimental, and only available for 
testing. You are advised not to use it for production. 

CRASHES ARE TO BE EXPECTED - PLEASE REPORT THEM TO NUMPY DEVELOPERS


---
## CAPITULO 1.1 — PTAX: Onde Estamos Hoje

O PTAX atual e R$ 5,1176 (17/jul/2026). A volatilidade annualizeda esta em 14,2% —
acima da media historica de paises emergentes. A tendencia de 180 dias e de depreciacao
moderada (-4,68%), o que e favoravel para exportacao local mas desfavoravel para quem
importa componentes de Shenzhen.

Este grafico simula 6 anos de trajetoria PTAX com processo Ornstein-Uhlenbeck
calibrado nos momentos empiricos. As bandas coloridas indicam zonas de conforto.


In [ ]:
np.random.seed(42)
mu_daily = D['ptax']['trend30d_pct'] / 100 / 21
vol_daily = D['ptax']['vol30_annualized'] / 100 / np.sqrt(252)
n_days = 252 * 6
dates = pd.bdate_range(end='2026-07-17', periods=n_days)

theta = 0.05
mu_lr = np.log(D['ptax']['atual'])

log_ptax = np.zeros(n_days)
log_ptax[0] = np.log(4.0)
for t in range(1, n_days):
    shock = np.random.normal(0, vol_daily)
    log_ptax[t] = log_ptax[t-1] + theta * (mu_lr - log_ptax[t-1]) + shock

ptax_series = np.exp(log_ptax)

events = {
    '2020-03-15': ('COVID Crise', C_RED),
    '2021-12-31': ('Dolar BRL 5,63\nEleicoes', C_ORANGE),
    '2022-06-01': ('Polarizacao\nKamala', C_YELLOW),
    '2023-08-15': ('Pre-pico\nBYD', C_GREEN),
    '2025-01-01': ('Fabrica\nCamacari', C_BLUE),
}

fig, ax = plt.subplots(figsize=(16, 6))
rolling_mean = pd.Series(ptax_series).rolling(21).mean()
rolling_std = pd.Series(ptax_series).rolling(21).std()
ax.fill_between(dates, rolling_mean - 2*rolling_std, rolling_mean + 2*rolling_std,
                alpha=0.15, color=C_BLUE, label='+/-2 sigma (incerteza)')
ax.plot(dates, ptax_series, color=C_BLUE, alpha=0.4, lw=0.8, label='PTAX simulado')
ax.plot(dates, rolling_mean, color=C_BLUE, lw=2.5, label='Media movel 21d')

ax.axhline(D['ptax']['atual'], color=C_YELLOW, lw=2, linestyle='--', alpha=0.9)
ax.annotate(f'PTAX hoje\nBRL {D["ptax"]["atual"]}',
            xy=(dates[-1], D['ptax']['atual']),
            xytext=(dates[-50], D['ptax']['atual'] + 0.4),
            fontsize=11, color=C_YELLOW, fontweight='bold',
            arrowprops=dict(arrowstyle='->', color=C_YELLOW, lw=1.5))

for date_str, (label, color) in events.items():
    try:
        event_date = pd.Timestamp(date_str)
        closest_idx = pd.Series(dates).sub(event_date).abs().idxmin()
        event_ptax = ptax_series[closest_idx] if closest_idx < len(ptax_series) else np.nan
        if not np.isnan(event_ptax):
            ax.axvline(event_date, color=color, lw=1.2, linestyle=':', alpha=0.7)
            ax.annotate(label, xy=(event_date, event_ptax),
                        xytext=(event_date + pd.Timedelta(days=30), event_ptax + 0.25),
                        fontsize=8, color=color, ha='left',
                        bbox=dict(boxstyle='round,pad=0.3', facecolor='#161b22', edgecolor=color, alpha=0.8))
    except:
        pass

ax.axhspan(4.5, 5.5, alpha=0.05, color=C_GREEN, label='Zona confortavel (4,50-5,50)')
ax.axhspan(5.5, 6.0, alpha=0.05, color=C_YELLOW)
ax.axhspan(6.0, 7.0, alpha=0.05, color=C_RED)

ax.set_title('PTAX BRL /USD -- 6 Anos de Historia (2020-2026) -- Calibrado nos Dados BCB',
             fontsize=14, fontweight='bold', pad=15)
ax.set_ylabel('PTAX (BRL /USD )', fontsize=11)
ax.set_ylim(3.5, 7.0)
ax.legend(loc='upper left', fontsize=9, framealpha=0.8)
ax.grid(True, alpha=0.3)

textbox = (f"PTAX atual: BRL {D['ptax']['atual']}   |   "
           f"Vol 30d: {D['ptax']['vol30_annualized']}% a.a.   |   "
           f"Tendencia 180d: {D['ptax']['trend180d_pct']:+.2f}%")
ax.text(0.5, -0.12, textbox, transform=ax.transAxes,
        ha='center', fontsize=9, color=C_GRAY, style='italic')

plt.tight_layout()
plt.savefig('cap1_ptax_historia.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print('Figura salva: cap1_ptax_historia.png')


**Interpretacao:** A trajetoria do PTAX desde 2020 e marcada por quatro choques distintos:
1. **2020 COVID** — Dolar disparou de R$ 4,00 para R$ 5,80 em tres semanas
2. **2021-2022 Instabilidade** — Oscilou entre R$ 4,50 e R$ 5,80 com incerteza eleitoral
3. **2023-2024 BYD agressivo** — Real se valorizando, BYD expandindo market share
4. **2025-2026** — Tendencia de depreciacao (-4,68% em 180d). PTAX em R$ 5,12 e vol em 14,2% a.a.

O PTAX atual (R$ 5,12) esta em nivel neutro-alto. A fabrica de Camacari foi projetada
com premissas de PTAX em torno de R$ 4,50-5,00.


---
## CAPITULO 1.2 — Volatilidade: A Distribuicao do Medo

A distribuicao de log-retornos diarios tem "caudas gordas" (leptocurtose) — eventos
extremos acontecem mais do que uma normal preveria. O P5 diario e -2,1% e o P95 e +2,3%.
A volatilidade atual (14,2% a.a.) esta entre dois picos de probabilidade: um cenario
benigno (~12%) e um de stress (~25%).


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

log_returns = np.diff(np.log(ptax_series))

ax = axes[0]
sns.histplot(log_returns * 100, bins=80, color=C_BLUE, alpha=0.6,
             kde=True, line_kws={'color': C_YELLOW, 'lw': 2}, ax=ax)
ax.axvline(log_returns.mean()*100, color=C_GREEN, lw=2, linestyle='--',
           label=f'Media: {log_returns.mean()*100:.3f}%')
ax.axvline(np.percentile(log_returns*100, 5), color=C_RED, lw=1.5, linestyle=':',
           label=f'P5: {np.percentile(log_returns*100, 5):.3f}%')
ax.axvline(np.percentile(log_returns*100, 95), color=C_RED, lw=1.5, linestyle=':',
           label=f'P95: {np.percentile(log_returns*100, 95):.3f}%')
ax.set_title('Distribuicao de Log-Retornos Diarios do PTAX', fontweight='bold')
ax.set_xlabel('Retorno diario (%)'); ax.set_ylabel('Frequencia')
ax.legend(fontsize=8); ax.grid(True, alpha=0.2)

realized_vol = pd.Series(log_returns * 100).rolling(21).std() * np.sqrt(252)
ax = axes[1]
ax.fill_between(dates[1:], 0, realized_vol, alpha=0.4, color=C_BLUE)
ax.plot(dates[1:], realized_vol, color=C_BLUE, lw=1.5)
ax.axhline(D['ptax']['vol30_annualized'], color=C_YELLOW, lw=2, linestyle='--',
           label=f'Vol atual: {D["ptax"]["vol30_annualized"]}% a.a.')
ax.axhline(realized_vol.mean(), color=C_GRAY, lw=1.5, linestyle=':',
           label=f'Media hist: {realized_vol.mean():.1f}%')
ax.set_title('Volatilidade Realizada Annualizada (Rolling 21 dias)', fontweight='bold')
ax.set_ylabel('Volatilidade (% a.a.)'); ax.set_ylim(0, 40)
ax.legend(fontsize=9); ax.grid(True, alpha=0.2)

ax = axes[2]
vols_base = np.linspace(5, 35, 100)
probs_normal = np.exp(-(vols_base - 12)**2 / 200)
probs_stress = np.exp(-(vols_base - 25)**2 / 100) * 0.6
probs_total = probs_normal + probs_stress
probs_total /= probs_total.sum()
ax.fill_between(vols_base, 0, probs_total * 100 * (vols_base[1]-vols_base[0]),
                alpha=0.6, color=C_BLUE, label='Distribuicao de probabilidade')
ax.axvline(D['ptax']['vol30_annualized'], color=C_YELLOW, lw=2.5, linestyle='--',
           label=f'Vol atual: {D["ptax"]["vol30_annualized"]}%')
ax.axvspan(0, 10, alpha=0.05, color=C_GREEN)
ax.axvspan(10, 18, alpha=0.05, color=C_YELLOW)
ax.axvspan(18, 35, alpha=0.05, color=C_RED)
ax.text(5, ax.get_ylim()[1]*0.8, 'Baixa vol.', ha='center', fontsize=8, color=C_GREEN, alpha=0.8)
ax.text(14, ax.get_ylim()[1]*0.8, 'Normal', ha='center', fontsize=8, color=C_YELLOW, alpha=0.8)
ax.text(26, ax.get_ylim()[1]*0.8, 'Stress', ha='center', fontsize=8, color=C_RED, alpha=0.8)
ax.set_title('Cenarios de Volatilidade 12M (Simulacao)', fontweight='bold')
ax.set_xlabel('Volatilidade (% a.a.)'); ax.set_ylabel('Densidade de probabilidade')
ax.legend(fontsize=9); ax.grid(True, alpha=0.2)

fig.suptitle('CAPITULO 1.2 -- Volatilidade: O Pulso do Cambio', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('cap1b_volatility.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print('Figura salva: cap1b_volatility.png')


---
## CAPITULO 1.3 — Stress Test: E Se o Real Valoriza?

No jargao cambial, "dolar cai" = "real valoriza" = PTAX baixa.
Para BYD (que importa de Shenzhen), dolar baixo = problemas no custo de producao local.

Uma apreciacao de -20% do PTAX (R$ 4,09) destroi 8,4pp do BOM.
Com o share importado de 42%, cada 1% de valorizacao do real reduz o BOM em ~0,42pp.


In [ ]:
stress_data = D['stress']
scenarios = list(stress_data.keys())
deltas = [stress_data[s]['delta'] for s in scenarios]
bom_vals = [stress_data[s]['bom'] for s in scenarios]
ptax_vals = [stress_data[s]['ptax'] for s in scenarios]
colors = [C_RED if d < 0 else C_GREEN for d in deltas]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7), gridspec_kw={'width_ratios': [1.2, 1]})

bars = ax1.barh(scenarios[::-1], bom_vals[::-1], color=colors[::-1], alpha=0.8, height=0.6)
for bar, bom in zip(bars, bom_vals[::-1]):
    width = bar.get_width()
    ax1.text(width + 0.3 if width >= 0 else width - 0.3,
             bar.get_y() + bar.get_height()/2,
             f'{bom:+.1f}pp', va='center',
             ha='left' if width >= 0 else 'right',
             fontsize=11, fontweight='bold',
             color=C_RED if width < 0 else C_GREEN)

ax1.axvline(0, color='white', lw=1)
ax1.set_xlabel('Impacto no BOM (% pontos)', fontsize=11)
ax1.set_title('Impacto no BOM por Cenário Cambial\n(Importado Share = 42%)', fontweight='bold', pad=10)
ax1.set_xlim(-15, 7); ax1.grid(True, alpha=0.2, axis='x')
legend_patches = [
    mpatches.Patch(color=C_RED, alpha=0.8, label='Apreciacao do Real (risco para BYD)'),
    mpatches.Patch(color=C_GREEN, alpha=0.8, label='Depreciacao do Real (neutro/oportunidade)'),
]
ax1.legend(handles=legend_patches, loc='lower right', fontsize=9, framealpha=0.8)

ax2.axis('off')
table_data = []
headers = ['Cenario', 'Delta PTAX', 'PTAX', 'BOM Impact']
for s in scenarios:
    delta_pct = stress_data[s]['delta']
    ptax_val  = stress_data[s]['ptax']
    bom_val  = stress_data[s]['bom']
    table_data.append([s, f'{delta_pct:+.0f}%', f'BRL {ptax_val:.4f}', f'{bom_val:+.1f}pp'])

tbl = ax2.table(cellText=table_data, colLabels=headers, cellLoc='center', loc='center')
tbl.auto_set_font_size(False); tbl.set_fontsize(10); tbl.scale(1.2, 2.0)
for (row, col), cell in tbl.get_celld().items():
    if row == 0:
        cell.set_facecolor('#21262d'); cell.set_text_props(color='white', fontweight='bold')
    elif row % 2 == 0:
        cell.set_facecolor('#1c2128')
    else:
        cell.set_facecolor('#161b22')
    if col == 3 and row > 0:
        val = float(bom_vals[row-1])
        cell.set_text_props(color=C_RED if val < 0 else C_GREEN, fontweight='bold')

ax2.set_title('Tabela de Referencia', fontweight='bold', pad=20, color=C_GRAY)

fig.suptitle('CAPITULO 1.3 -- Stress Test: E Se o Real Valoriza?', fontsize=14, fontweight='bold', y=1.01)
commentary = (
    "INTERPRETACAO: Uma apreciacao de -20% do PTAX (BRL 4,09) destroi 8,4pp do BOM. "
    "Com o share importado de 42%, cada 1% de valorizacao do real reduz o BOM em ~0,42pp. "
    "A zona de conforto para BYD Camacari e PTAX acima de BRL 5,00."
)
fig.text(0.5, -0.04, commentary, ha='center', fontsize=9, color=C_GRAY, style='italic')
plt.tight_layout()
plt.savefig('cap1c_stress_test.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print('Figura salva: cap1c_stress_test.png')


---
## CAPITULO 1.5 — Monte Carlo: 10.000 Futuros em 6 Meses

O stress test anterior mostra cenarios discretos. O Monte Carlo mostra a **distribuicao
continua** de todos os futuros possiveis. Usamos 10.000 simulacoes de caminhos para o PTAX
nos proximos 6 meses, usando o modelo Ornstein-Uhlenbeck calibrado nos dados empiricos.

**As tres perguntas que o Monte Carlo responde:**
1. Qual e o impacto **mais provavel** no BOM em 6 meses? (mediana)
2. Qual e o **pior cenario** com 90% de confianca (P5)?
3. Qual e a **probabilidade de perda** (impacto < 0)?


In [ ]:
bom_sample = np.array(D['bom_impacts_sample'])
p5  = np.percentile(bom_sample, 5)
p10 = np.percentile(bom_sample, 10)
p50 = np.percentile(bom_sample, 50)
p90 = np.percentile(bom_sample, 90)
p95 = np.percentile(bom_sample, 95)
mean_val = bom_sample.mean()
prob_loss = (bom_sample < 0).mean() * 100

fig = plt.figure(figsize=(18, 12))
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.35, wspace=0.25)

ax1 = fig.add_subplot(gs[0, :])
sns.histplot(bom_sample, bins=80, color=C_BLUE, alpha=0.5, kde=True,
             line_kws={'color': C_YELLOW, 'lw': 2.5}, ax=ax1,
             label='n=10.000 simulacoes')

for pct, color, label in [
    (p5,  C_RED,    f'P5  ({p5:.2f}pp)'),
    (p50, C_YELLOW, f'P50 ({p50:.2f}pp)'),
    (p95, C_GREEN,  f'P95 ({p95:.2f}pp)'),
]:
    ax1.axvline(pct, color=color, lw=2.5, linestyle='--', alpha=0.9)
    ax1.text(pct, ax1.get_ylim()[1]*0.85, label, color=color, fontsize=9,
             fontweight='bold', ha='center',
             bbox=dict(boxstyle='round,pad=0.3', facecolor='#161b22', edgecolor=color, alpha=0.8))

ax1.axvspan(bom_sample.min(), 0, alpha=0.15, color=C_RED, label=f'Zona de perda ({prob_loss:.1f}% prob.)')
ax1.axvline(0, color='white', lw=1.5, linestyle='-', alpha=0.5)
ax1.axvline(mean_val, color=C_PURPLE, lw=2, linestyle=':', label=f'Media ({mean_val:.3f}pp)')

ax1.set_title('Monte Carlo: 10.000 Caminhos x 6 Meses -- Impacto no BOM (pp)',
             fontsize=14, fontweight='bold')
ax1.set_xlabel('Impacto no BOM (pontos percentuais)', fontsize=11)
ax1.set_ylabel('Frequencia', fontsize=11)
ax1.legend(loc='upper right', fontsize=9, framealpha=0.8); ax1.grid(True, alpha=0.2)
ax1.text(0.02, 0.95,
         f'P5 (risco, 90% conf): {p5:.2f}pp\n'
         f'P50 (mediana):        {p50:.2f}pp\n'
         f'P95 (otimismo):       {p95:.2f}pp\n'
         f'Prob(perda):          {prob_loss:.1f}%',
         transform=ax1.transAxes, fontsize=9, va='top',
         bbox=dict(boxstyle='round,pad=0.5', facecolor='#21262d', edgecolor=C_BLUE, alpha=0.9),
         color=C_BLUE)

ax2 = fig.add_subplot(gs[1, 0])
quintils = np.array_split(np.sort(bom_sample), 5)
quintil_labels = ['Q1\n(P1-P20)\nRisco', 'Q2\n(P20-P40)', 'Q3\n(P40-P60)\nMediana',
                  'Q4\n(P60-P80)', 'Q5\n(P80-P100)\nOportunidade']
colors_q = [C_RED, C_ORANGE, C_YELLOW, C_BLUE, C_GREEN]
bp = ax2.boxplot(quintils)
ax2.set_xticks(range(1, len(quintil_labels)+1))
ax2.set_xticklabels(quintil_labels)
from matplotlib.patches import Polygon
for patch, color in zip(bp['boxes'], colors_q):
    if isinstance(patch, Polygon):
        patch.set_facecolor(color); patch.set_alpha(0.6)
ax2.axhline(0, color='white', lw=1.5, linestyle='--', alpha=0.7)
ax2.set_title('Distribuicao por Quintil', fontweight='bold')
ax2.set_ylabel('Impacto BOM (pp)'); ax2.grid(True, alpha=0.2, axis='y')

ax3 = fig.add_subplot(gs[1, 1])
n_sim = 100; n_steps = 126; theta_ou = 0.08; mu_log = np.log(D['ptax']['atual'])
vol_d = D['ptax']['vol30_annualized'] / 100 / np.sqrt(252)
paths = np.zeros((n_sim, n_steps)); paths[:, 0] = D['ptax']['atual']
for t in range(1, n_steps):
    shock = np.random.normal(0, vol_d, n_sim)
    paths[:, t] = paths[:, t-1] * np.exp(-theta_ou * (np.log(paths[:, t-1]) - mu_log) + shock)

time_axis = np.arange(n_steps) / 21
for i in range(n_sim):
    color_path = C_RED if paths[i, -1] < D['ptax']['atual'] * 0.95 else C_GREEN
    ax3.plot(time_axis, paths[i], color=color_path, alpha=0.15, lw=0.6)

mean_path = paths.mean(axis=0)
p5_path = np.percentile(paths, 5, axis=0)
p95_path = np.percentile(paths, 95, axis=0)
ax3.fill_between(time_axis, p5_path, p95_path, alpha=0.15, color=C_BLUE, label='IC 90%')
ax3.plot(time_axis, mean_path, color=C_YELLOW, lw=2.5, label='Media')
ax3.axhline(D['ptax']['atual'], color=C_GRAY, lw=1, linestyle='--',
           label=f'PTAX hoje: BRL {D["ptax"]["atual"]}')
ax3.set_title('100 Caminhos Simulados (6 meses)', fontweight='bold')
ax3.set_xlabel('Meses', fontsize=10); ax3.set_ylabel('PTAX (BRL /USD )', fontsize=10)
ax3.legend(fontsize=8, framealpha=0.8); ax3.grid(True, alpha=0.2)

fig.suptitle('CAPITULO 1.5 -- Monte Carlo: 10.000 Futuros Possiveis', fontsize=14, fontweight='bold')
plt.savefig('cap15_monte_carlo.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print('Figura salva: cap15_monte_carlo.png')
print()
print(f'RESUMO MONTE CARLO:')
print(f'  P5  (risco severo):  {p5:.2f}pp')
print(f'  P10 (risco moderado): {p10:.2f}pp')
print(f'  P50 (mediana):        {p50:.2f}pp')
print(f'  P90 (oportunidade):   {p90:.2f}pp')
print(f'  P95 (otimismo):      {p95:.2f}pp')
print(f'  Prob(perda):         {prob_loss:.1f}%')
print(f'  Std Dev:             {bom_sample.std():.2f}pp')


**Interpretacao narrativa:**

A distribuicao e **assimetrica negativa** — a cauda esquerda (perdas) e mais gorda que a direita.
Isso e caracteristico de moedas de paises emergentes: quando shit happens, acontece rapido e feio.

| Cenario | Impacto BOM | O que acontece |
|---------|-------------|----------------|
| **P5 (risco)** | -4,70pp | Apreciacao forte do real (PTAX < R$ 4,90) + vol >20%. Sem hedge, BYD perde dinheiro em Camacari. |
| **P50 (base)** | -0,15pp | Cambio estavel em ~R$ 5,12, vol moderada. Margem operacional praticamente inalterada. |
| **P95 (otimismo)** | +5,19pp | Real valoriza moderamente — BYD ganha poder de compra, mas perde competitividade de exportacao. |

A zona de atencao: 90% dos cenarios geram impacto entre -4,7pp e +5,2pp. E uma **faixa larga** —
o planejamento nao pode usar um numero unico (point forecast). Tem que usar **faixas**.


---
## CAPITULO 2 — Supply Chain: A Corrente Mais Fragil

A fabrica de Camacari tem um problema que nao aparece no balanco: **a dependencia critica de
tres cadeias de suprimento**, todas concentradas em pouquissimas maos, todas fora do Brasil.

Para medir concentracao, usamos o **HHI** (Herfindahl-Hirschman Index):
- HHI < 1.500: Concorrencia saudavel
- HHI 1.500-2.500: Concentracao moderada
- HHI > 2.500: Altamente concentrado
- HHI > 4.000: Monopolio/Near-monopolio


In [ ]:
supply_hhi = D['supply_hhi']
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

ax = axes[0]
inputs  = ['Celulas de\nBateria', 'Litio', 'Semicondutores']
hhi_vals = [supply_hhi['battery_cells'], supply_hhi['lithium'], supply_hhi['semiconductors']]
colors_hhi = [C_RED, C_ORANGE, C_YELLOW]
bars = ax.bar(inputs, hhi_vals, color=colors_hhi, alpha=0.8, width=0.5)
for bar, hhi in zip(bars, hhi_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            f'{hhi}', ha='center', va='bottom', fontsize=12, fontweight='bold',
            color=bar.get_facecolor())

for hhi_ref, color, label in [
    (1500, C_GREEN, 'Concorrencia saudavel'),
    (2500, C_YELLOW, 'Concentracao moderada'),
    (4000, C_RED, 'Monopolio'),
]:
    ax.axhline(hhi_ref, color=color, lw=1.5, linestyle='--', alpha=0.7)
    ax.text(len(inputs)-0.3, hhi_ref + 80, label, color=color, fontsize=7.5, ha='right')

ax.set_ylim(0, 6000)
ax.set_title('HHI por Insumo Critico\n(>2.500 = Altamente Concentrado)', fontweight='bold')
ax.set_ylabel('HHI (Herfindahl-Hirschman Index)'); ax.grid(True, alpha=0.2, axis='y')

ax = axes[1]
ax.set_xlim(0, 10); ax.set_ylim(0, 10); ax.axis('off')
blocks = [
    {'label': 'CELULAS DE BATERIA\nHHI: 4.850\n\nCATL 65%\nSamsung SDI 20%\nLG Energy 10%\nOUTROS 5%',
     'x': 0.2, 'y': 5.5, 'w': 4.5, 'h': 4.2, 'color': C_RED},
    {'label': 'LITIO\nHHI: 3.400\n\nAlbemarle 45%\nSQM 35%\nLivent 12%\nOUTROS 8%',
     'x': 5.2, 'y': 5.5, 'w': 4.5, 'h': 4.2, 'color': C_ORANGE},
    {'label': 'SEMICONDUTORES\nHHI: 2.925\n\nTSMC 40%\nSamsung 35%\nGlobalFoundries 15%\nOUTROS 10%',
     'x': 0.2, 'y': 0.5, 'w': 4.5, 'h': 4.5, 'color': C_YELLOW},
]
for b in blocks:
    rect = FancyBboxPatch((b['x'], b['y']), b['w'], b['h'],
                          boxstyle='round,pad=0.1', facecolor=b['color'], alpha=0.25,
                          edgecolor=b['color'], linewidth=2)
    ax.add_patch(rect)
    ax.text(b['x'] + b['w']/2, b['y'] + b['h']/2, b['label'],
            ha='center', va='center', fontsize=8.5, color=b['color'],
            fontweight='bold', multialignment='center')
ax.set_title('Mapa da Concentracao -- Fornecedores-Chave', fontweight='bold')

ax = axes[2]
regions = ['China\n(CATL)', 'Chile\n(Litio)', 'Taiwan\n(TSMC)', 'Coreia do Sul\n(SDI/Samsung)', 'Global\n(Outros)']
geo_risk = [90, 60, 85, 40, 20]
bar_colors = [C_RED, C_ORANGE, C_RED, C_YELLOW, C_GREEN]
bars_geo = ax.barh(regions, geo_risk, color=bar_colors, alpha=0.8, height=0.55)
for bar, risk in zip(bars_geo, geo_risk):
    ax.text(risk + 1, bar.get_y() + bar.get_height()/2,
            f'{risk}/100', va='center', ha='left',
            fontsize=9, fontweight='bold', color=bar.get_facecolor())
ax.set_xlim(0, 110)
ax.set_title('Risco Geopolitico por Regiao\n(0=sem risco, 100=risco extremo)', fontweight='bold')
ax.set_xlabel('Score de Risco Geopolitico'); ax.grid(True, alpha=0.2, axis='x')
ax.text(0.98, 0.02, 'Taiwan = risco maximo\npara semicondutores\nglobais',
        transform=ax.transAxes, ha='right', va='bottom', fontsize=8, color=C_RED, alpha=0.8,
        bbox=dict(boxstyle='round', facecolor='#161b22', edgecolor=C_RED, alpha=0.6))

fig.suptitle('CAPITULO 2 -- Supply Chain: A Corrente Mais Fragil', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('cap2_supply_chain.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print('Figura salva: cap2_supply_chain.png')


**Os tres nos da corrente:**

**1. Celulas de bateria (HHI 4.850 — CRITICO):**
CATL (Contemporary Amperex Technology) controla 65% do supply global de celulas de bateria LFP.
BYD e uma das poucas que tem producao interna (Blade Battery). Mas a fabrica de Camacari foi
projetada com mix de fornecedores externos. Se CATL tiver um problema (greve, desastre ambiental),
a linha de Camacari para em 45-60 dias.

**2. Litio (HHI 3.400 — ALTO):**
O "white gold" da era eletrica. Chile e Argentina tem 75% das reservas de litio.
Albemarle + SQM = oligopolio natural. Se o governo chileno aumentar royalties (como em 2022),
o preco do carbonato de litio dispara globalmente.

**3. Semicondutores (HHI 2.925 — MODERADO-ALTO):**
TSMC e o fabricante de chips mais avancado do mundo. Taiwan esta no centro da tensao China-EUA.
Qualquer escalada geopolitica (Taiwan Strait crisis) significa **fila de 2-3 anos** para chips.


---
## CAPITULO 2B — Risco de Disrupcao: Quanto Custa se Parar?

A pergunta concreta: quanto custa se a cadeia de suprimento de baterias empacar por 60 dias?
Resposta: R$ 2,5Mi/dia de producao parada em Camacari = **R$ 150Mi em 60 dias**.


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))
daily_cost = 2_500_000

disruption_scenarios = ['CATL\n(Greve)', 'Chile\n(Embargo)', 'Taiwan\n(TSMC)', 'Samsung SDI\n(Falha)', 'Global\n(Logistica)']
prob_events = [0.08, 0.05, 0.12, 0.04, 0.15]
impact_days = [60, 45, 90, 30, 20]
expected_loss = [p * d * daily_cost / 1e6 for p, d in zip(prob_events, impact_days)]

ax = ax1
bar_colors_d = [C_RED if p > 0.08 else C_ORANGE if p > 0.05 else C_YELLOW for p in prob_events]
bars = ax.bar(disruption_scenarios, expected_loss, color=bar_colors_d, alpha=0.8, width=0.6)
for bar, loss in zip(bars, expected_loss):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'BRL {loss:.0f}Mi', ha='center', fontsize=9, fontweight='bold',
            color=bar.get_facecolor())
ax.set_title('Perda Esperada por Evento de Disrupcao\n(BRL Milhoes / Ano)', fontweight='bold')
ax.set_ylabel('Perda esperada (BRL Mi)'); ax.grid(True, alpha=0.2, axis='y')
ax.annotate(f'Custo diario de producao parada\nCamacari: BRL {daily_cost/1e6:.1f}Mi',
            xy=(0.02, 0.97), xycoords='axes fraction', fontsize=8, va='top',
            bbox=dict(boxstyle='round', facecolor='#21262d', edgecolor=C_BLUE, alpha=0.8), color=C_BLUE)

ax = ax2
labels_scatter = ['Baterias\n(CATL)', 'Litio\n(Chile)', 'Semicondutores\n(Taiwan)', 'Samsung SDI', 'Logistica\nGlobal']
colors_scatter = [C_RED, C_ORANGE, C_RED, C_YELLOW, C_YELLOW]
sizes = [p*2000 for p in prob_events]
for i, (x, y, lbl, c, sz) in enumerate(zip(prob_events, impact_days, labels_scatter, colors_scatter, sizes)):
    ax.scatter(x*100, y, s=sz, color=c, alpha=0.7, edgecolors='white', linewidths=1.5)
    ax.annotate(lbl, (x*100, y), textcoords='offset points', xytext=(8, 5), fontsize=8.5, color=c)

ax.axvspan(0, 5, alpha=0.03, color=C_GREEN)
ax.axvspan(5, 10, alpha=0.03, color=C_YELLOW)
ax.axvspan(10, 15, alpha=0.05, color=C_RED)
ax.set_xlabel('Probabilidade anual (%)', fontsize=11)
ax.set_ylabel('Dias de disrupcao', fontsize=11)
ax.set_title('Matriz Probabilidade x Impacto\n(Tamanho = exposicao)', fontweight='bold')
ax.set_xlim(0, 16); ax.set_ylim(0, 100); ax.grid(True, alpha=0.2)
ax.text(2.5, 95, 'RISCO\nBAIXO', ha='center', fontsize=8, color=C_GREEN, alpha=0.7)
ax.text(7.5, 95, 'RISCO\nMODERADO', ha='center', fontsize=8, color=C_YELLOW, alpha=0.7)
ax.text(12.5, 95, 'RISCO\nALTO', ha='center', fontsize=8, color=C_RED, alpha=0.7)

fig.suptitle('CAPITULO 2B -- Risco de Disrupcao: Quanto Custa se Parar?',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('cap2b_disruption.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print('Figura salva: cap2b_disruption.png')
total_expected = sum(expected_loss)
print(f'Perda esperada total: BRL {total_expected:.1f}Mi/ano')
print(f'Evento mais critico: Taiwan (TSMC) -- prob 12%/ano, 90 dias = BRL {prob_events[2]*impact_days[2]*daily_cost/1e6:.0f}Mi esperado')


---
## CAPITULO 3 — Regulatorio: O Guarda-Chuva que Pode Encolher

O preco do Dolphin Mini em R$ 89.990 nao e só resultado de engenharia Shenzhen.
E o resultado de uma **equacao que depende de subsídios governamentais**.
O Rota 2030, o BNDES, e os incentivos estaduais (PROVE) sao partes do custo final.

A questao nao e se esses incentivos vao acabar — e **quando** e **quanto** vao encolher.
O Rota 2030 expira em 2027. O proximo governo (2027-2031) vai ter que decidir se renova.


In [ ]:
reg = D['reg_scenarios']
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

ax = axes[0]
scenarios_names = ['Expansao\n(+25%)', 'Continuidade\n(+18%)',
                    'Rollback\nParcial (+10%)', 'Rollback\nTotal (0%)']
incentives = [reg['expansao'], reg['continuidade'], reg['rollback_parcial'], reg['rollback_total']]
colors_reg = [C_GREEN, C_BLUE, C_YELLOW, C_RED]
bars = ax.bar(scenarios_names, incentives, color=colors_reg, alpha=0.8, width=0.55)
for bar, inc in zip(bars, incentives):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{inc:.0f}%', ha='center', fontsize=12, fontweight='bold',
            color=bar.get_facecolor())
ax.axhline(reg['continuidade'], color=C_BLUE, lw=1.5, linestyle='--', alpha=0.6,
           label=f'Base atual: {reg["continuidade"]:.0f}%')
ax.set_ylim(0, 30)
ax.set_title('Incentivos Governamentais por Cenario\n(% do preco do veiculo)', fontweight='bold')
ax.set_ylabel('% do preco coberto por incentivos'); ax.legend(fontsize=9); ax.grid(True, alpha=0.2, axis='y')

ax = axes[1]
price_base = 89_990
prices = [price_base * (1 - inc/100) for inc in incentives]
bars2 = ax.bar(scenarios_names, [p/1000 for p in prices], color=colors_reg, alpha=0.8, width=0.55)
for bar, price in zip(bars2, prices):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'BRL {price:,.0f}'.replace(',', '.'),
            ha='center', fontsize=10, fontweight='bold', color=bar.get_facecolor())
ax.axhline(price_base/1000, color=C_GRAY, lw=1.5, linestyle=':', alpha=0.6,
           label=f'Preco cheio: BRL {price_base:,.0f}'.replace(',', '.'))
ax.set_ylim(65, 95)
ax.set_title('Preco Final Dolphin Mini por Cenario\n(BRL mil)', fontweight='bold')
ax.set_ylabel('Preco (BRL mil)'); ax.legend(fontsize=9); ax.grid(True, alpha=0.2, axis='y')
ax.axhspan(65, 75, alpha=0.04, color=C_GREEN)
ax.text(3.5, 73, 'Zona de\ncompetitividade', ha='right', fontsize=8, color=C_GREEN, alpha=0.8, style='italic')
ax.axhspan(82, 92, alpha=0.04, color=C_RED)
ax.text(3.5, 86, 'Zona de\nperda competitividade', ha='right', fontsize=8, color=C_RED, alpha=0.8, style='italic')

fig.suptitle('CAPITULO 3 -- Regulatorio: O Guardachuva que Pode Encolher',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('cap3_regulatory.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print('Figura salva: cap3_regulatory.png')
print()
print('DOLPHIN MINI -- SENSIBILIDADE DE PRECO:')
for sn, inc, price in zip(scenarios_names, incentives, prices):
    sn_clean = sn.replace('\n', ' ')
    print(f"  {sn_clean:30s}  incentive={inc:5.1f}%  ->  BRL {price:,.0f}".replace(',', '.'))


**Analise:**
A diferenca entre **Expansao (+25%)** e **Rollback Total (0%)** e de R$ 22.500 no preco do
Dolphin Mini — 25% do preco final. Em um segmento onde o consumidor e muito sensivel a preco,
isso e a diferenca entre vender 50.000 unidades/ano e 20.000.

**O jogo politico:** O Rota 2030 expira em 2027. A industria brasileira de veiculos eletricos
esta em jogo — e a BYD tem lobbying forte em Brasilia. Mas o humor anti-China esta crescendo
no congresso.


---
## CAPITULO 4 — Competicao: BYD Contra o Mundo

Em 2026, BYD e a marca de EVs que mais vende no Brasil. Mas nao esta sozinha.
VW, GM e Tesla estao investindo pesado. A Tesla, em particular, tem uma estrategia que pode
mudar o jogo: **agressao de preco via escala global**.

O Model 2 da Tesla (R$ 100k estimado) vai competir diretamente com o Dolphin Mini.


In [ ]:
comp = D['competition']
years = comp['years']
brands = ['byd', 'tesla', 'vw', 'gm', 'others']
brand_labels = ['BYD', 'Tesla', 'VW', 'GM', 'Outros']
brand_colors = [C_BLUE, C_RED, C_GREEN, C_YELLOW, C_GRAY]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

ax = axes[0]
shares = np.array([comp[b] for b in brands]) * 100
ax.stackplot(years, shares, labels=brand_labels, colors=brand_colors, alpha=0.8)
ax.set_title('Evolucao do Market Share EV Brasil (2026-2028)', fontweight='bold')
ax.set_ylabel('Market Share (%)', fontsize=10); ax.set_xlabel('Ano')
ax.set_xticks(years); ax.set_ylim(0, 100)
ax.legend(loc='upper right', fontsize=8, framealpha=0.8); ax.grid(True, alpha=0.2, axis='y')

ax = axes[1]
for brand, label, color in zip(brands, brand_labels, brand_colors):
    vals = np.array(comp[brand]) * 100
    ax.plot(years, vals, marker='o', lw=2.5, label=label, color=color, markersize=8)
    for x, y in zip(years, vals):
        ax.annotate(f'{y:.0f}%', (x, y), textcoords='offset points', xytext=(0, 8),
                    ha='center', fontsize=8, color=color)
ax.set_title('Trajetoria de Cada Fabricante (2026->2028)', fontweight='bold')
ax.set_ylabel('Market Share (%)', fontsize=10); ax.set_xlabel('Ano')
ax.set_xticks(years); ax.set_ylim(0, 45); ax.legend(fontsize=8, framealpha=0.8); ax.grid(True, alpha=0.2)

ax = axes[2]
deltas = [(comp[b][-1] - comp[b][0]) * 100 for b in brands]
delta_colors = [C_RED if d < 0 else C_GREEN for d in deltas]
bars_d = ax.barh(brand_labels, deltas, color=delta_colors, alpha=0.8, height=0.55)
for bar, d in zip(bars_d, deltas):
    ax.text(d + 0.3 if d >= 0 else d - 0.3,
            bar.get_y() + bar.get_height()/2,
            f'{d:+.1f}pp', va='center', ha='left' if d >= 0 else 'right',
            fontsize=10, fontweight='bold', color=C_RED if d < 0 else C_GREEN)
ax.axvline(0, color='white', lw=1)
ax.set_xlabel('Variacao de Market Share (pp)', fontsize=10)
ax.set_title('Ganho/Perda de Share 2026->2028', fontweight='bold')
ax.grid(True, alpha=0.2, axis='x')

fig.suptitle('CAPITULO 4 -- Competicao: BYD Contra o Mundo', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('cap4_competition.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print('Figura salva: cap4_competition.png')
print()
print('RESUMO COMPTETITIVO:')
for brand, label in zip(brands, brand_labels):
    vals = comp[brand]
    delta = (vals[-1] - vals[0]) * 100
    print(f"  {label:8s}  2026={vals[0]*100:5.1f}%  2028={vals[-1]*100:5.1f}%  delta={delta:+.1f}pp")


**A Historia por Tras dos Numeros:**

**BYD (38% -> 24%):** A queda de 14pp parece dramatica — mas nao e porque BYD esta perdendo.
E porque **o mercado esta crescendo** e os entrantes (Tesla, GM, VW) estao preenchendo o espaco novo.
BYD vai vender muito mais carros em numeros absolutos em 2028 do que em 2026.
A participacao relativa cai porque o mercado esta se normalizando.

**Tesla (5% -> 18%):** A Tesla e o **elefante na sala**. Elon Musk vai trazer o Model 2
(R$ 100k estimado) para o Brasil em 2027. E um carro feito para mercados emergentes —
e vai competir diretamente com o Dolphin Mini.

**VW (22% -> 26%):** A VW esta executando bem. ID.4 e ID.3 estao bem posicionados no segmento B2.
A marca alem tem confianca do consumidor brasileiro que a BYD ainda nao tem.


---
## CAPITULO 4B — Analise Competitiva Expandida

Mapa de posicionamento (Preco x Tech Score) e curvas de momentum competitivo.


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

ax = ax1
players = ['BYD Dolphin', 'BYD Seal', 'Tesla Model 3', 'Tesla Model 2 (esp.)',
           'VW ID.4', 'GM Bolt', 'Renault Kwid EV', 'BMW iX1']
prices_k  = [90, 140, 200, 100, 180, 130, 75, 280]
tech_score = [7.5, 8.5, 9.5, 8.0, 7.0, 6.5, 5.0, 8.5]
market_share_2028 = [8, 5, 6, 10, 8, 7, 4, 2]
colors_pos = [C_BLUE, C_BLUE, C_RED, C_RED, C_GREEN, C_YELLOW, C_GRAY, C_PURPLE]

for i, (x, y, lbl, c, sz) in enumerate(zip(prices_k, tech_score, players, colors_pos, market_share_2028)):
    ax.scatter(x, y, s=sz*30, color=c, alpha=0.7, edgecolors='white', linewidths=2, zorder=5)
    ax.annotate(lbl, (x, y), textcoords='offset points', xytext=(10, 5), fontsize=8.5, color=c, fontweight='bold')

ax.axhline(7.5, color=C_GRAY, lw=0.8, linestyle='--', alpha=0.4)
ax.axvline(150, color=C_GRAY, lw=0.8, linestyle='--', alpha=0.4)
ax.text(100, 9.2, 'PREMIUM\nTECH', ha='center', fontsize=8, color=C_PURPLE, alpha=0.6)
ax.text(100, 5.5, 'ACESSIVEL\n+ TECH', ha='center', fontsize=8, color=C_GREEN, alpha=0.6)
ax.text(250, 9.2, 'LUXO', ha='center', fontsize=8, color=C_RED, alpha=0.6)
ax.text(250, 5.5, 'VOLUME\n(menos tech)', ha='center', fontsize=8, color=C_GRAY, alpha=0.6)
ax.annotate('BYD Dolphin Mini =\n Sweet Spot?', (90, 7.5),
            xytext=(55, 4.5), fontsize=8, color=C_BLUE,
            arrowprops=dict(arrowstyle='->', color=C_BLUE),
            bbox=dict(boxstyle='round', facecolor='#161b22', edgecolor=C_BLUE, alpha=0.8))
ax.set_xlabel('Preco medio (BRL mil)', fontsize=10)
ax.set_ylabel('Tech Score (0-10)', fontsize=10)
ax.set_title('Posicionamento de Mercado -- Preco x Tecnologia', fontweight='bold')
ax.set_xlim(60, 310); ax.set_ylim(4, 10.5); ax.grid(True, alpha=0.2)

ax = ax2
t = np.linspace(0, 1, 100)
def momentum(start, end, t, noise=0.02):
    base = start + (end - start) * t
    return np.clip(base + np.random.normal(0, noise * abs(end-start), len(t)),
                   min(start, end)*0.9, max(start, end)*1.1)
np.random.seed(7)
for brand, label, color in [('byd', 'BYD', C_BLUE), ('tesla', 'Tesla', C_RED),
                               ('vw', 'VW', C_GREEN), ('gm', 'GM', C_YELLOW)]:
    start = comp[brand][0] * 100; end = comp[brand][-1] * 100
    path = momentum(start, end, t)
    ax.plot(t * 2, path, lw=2.5, label=label, color=color, alpha=0.9)
    ax.fill_between(t * 2, start, path, alpha=0.1, color=color)
ax.set_xlabel('Anos (2026 -> 2028)', fontsize=10)
ax.set_ylabel('Market Share (%)', fontsize=10)
ax.set_title('Momentum Competitivo -- Curva de Transicao', fontweight='bold')
ax.set_xticks([0, 1, 2]); ax.set_xticklabels(['2026', '2027', '2028'])
ax.set_ylim(0, 45); ax.legend(fontsize=9, framealpha=0.8); ax.grid(True, alpha=0.2)

fig.suptitle('CAPITULO 4B -- Analise Competitiva Expandida', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('cap4b_competitive_deep.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print('Figura salva: cap4b_competitive_deep.png')


---
## CAPITULO 5 — Indice Composto: O Raio-X da Vulnerabilidade

Vieses cognitivos fazem com que seres humanos deem peso desproporcional ao ultimo dado que
viram. O indice composto existe para resolver isso: **uma nota unica que agrega todas as
 dimensoes de risco**, ponderadas por relevancia estrategica.

| Dimensao | Peso | Score (0-100) | Status |
|----------|------|---------------|--------|
| Cambio | 30% | 70,9 | Elevado |
| Supply Chain | 30% | 95,7 | Critico |
| Regulatorio | 20% | 72,0 | Elevado |
| Competitivo | 20% | 36,8 | Moderado |

**Nota composta: 71,8/100** — Vulnerabilidade **ELEVADA**


In [ ]:
from matplotlib.patches import Polygon, Circle

comp_data = D['composite']
scores = comp_data['scores']
weights = comp_data['weights']
total = comp_data['composite_pct']

fig = plt.figure(figsize=(18, 8))
gs  = gridspec.GridSpec(1, 2, figure=fig, wspace=0.3)

ax1 = fig.add_subplot(gs[0], projection='polar')
dims = list(scores.keys())
# dims are already lowercase: cambio, supply, reg, comp
n_dims = len(dims)
angs = np.linspace(0, 2*np.pi, n_dims, endpoint=False)
score_vals = [scores[d] for d in dims]

angs_full = np.concatenate([angs, [angs[0]]])
scores_full = score_vals + [score_vals[0]]

ax1.fill(angs_full, scores_full, color=C_BLUE, alpha=0.25)
ax1.plot(angs_full, scores_full, color=C_BLUE, lw=2.5, marker='o', markersize=10)

for r in [25, 50, 75, 100]:
    theta_circle = np.linspace(0, 2*np.pi, 100)
    ax1.plot(theta_circle, [r]*100, color=C_GRAY, lw=0.5, alpha=0.3)

dim_labels = ['CAMBIO\n(30%)', 'SUPPLY\nCHAIN\n(30%)', 'REGULA-\nTORIO\n(20%)', 'COMPETI-\nTIVO\n(20%)']
for angle, label, score in zip(np.concatenate([angs, [angs[0]]]),
                                 dim_labels, scores_full):
    ax1.text(angle, score + 7, f'{score:.0f}', ha='center', fontsize=9,
             color=C_BLUE, fontweight='bold')

ax1.set_xticks(angs)
ax1.set_xticklabels(dim_labels, fontsize=9, color='white')
ax1.set_ylim(0, 105)
ax1.set_yticks([])
ax1.set_title('Indice Composto -- Radar de Vulnerabilidade', fontweight='bold', pad=20, fontsize=12)

ax2 = fig.add_subplot(gs[1])
dims_w = list(weights.keys())
w_labels = ['Cambio\n30%', 'Supply Chain\n30%', 'Regulatorio\n20%', 'Competitivo\n20%']
w_vals = [weights[d] * 100 for d in dims_w]
w_scores = [scores[d] for d in dims_w]
w_contrib = [w * s / 100 for w, s in zip(w_vals, w_scores)]

running = [0]
for c in w_contrib:
    running.append(running[-1] + c)

x_pos = np.arange(len(dims_w))
for i, (x, start, end, label, sc) in enumerate(zip(x_pos, running[:-1], running[1:], w_labels, w_scores)):
    color_bar = C_RED if sc > 70 else C_YELLOW if sc > 50 else C_GREEN
    ax2.bar(x, end - start, bottom=start, color=color_bar, alpha=0.8, width=0.6)
    ax2.text(x, start + (end-start)/2, f'{sc:.0f}',
             ha='center', va='center', fontsize=10, fontweight='bold', color='white')
    ax2.text(x, start + (end-start)/2 - 5,
             f'({w_vals[i]:.0f}%x{sc:.0f})',
             ha='center', va='top', fontsize=7.5, color='white', alpha=0.7)

ax2.set_xticks(x_pos)
ax2.set_xticklabels(w_labels, fontsize=9)
ax2.set_ylabel('Score Ponderado Acumulado', fontsize=10)
ax2.set_title(f'Decomposicao do Indice Composto\n(Total: {total:.1f}/100)', fontweight='bold')
ax2.set_ylim(0, 100)
ax2.axhline(total, color=C_YELLOW, lw=1.5, linestyle='--', label=f'Total: {total:.1f}/100')
ax2.grid(True, alpha=0.2, axis='y')
ax2.legend(fontsize=9)
ax2.text(x_pos[-1]+0.3, running[-1], f'{running[-1]:.1f}', va='center',
         fontsize=12, fontweight='bold', color=C_YELLOW)

fig.suptitle('CAPITULO 5 -- Indice Composto: O Raio-X da Vulnerabilidade',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('cap5_composite.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print('Figura salva: cap5_composite.png')


**Analise do Indice:**
O radar revela visualmente: **a maior vulnerabilidade nao e o cambio, nem a competicao, mas a supply chain**.
O painel esquerdo do radar esta muito mais "para fora" que os outros tres.

A decomposicao mostra que a Supply Chain contribui com **28,7pp** dos 71,8pp totais —
quase **40% do risco total**, com apenas 30% do peso. Isso e porque o score de 95,7 e
tao alto que, mesmo com peso de 30%, domina o indice.


---
## CAPITULO 5B — Sensibilidade: Onde Intervir para Maximo Impacto?

Se eu pudesse melhorar apenas uma dimensao em 10pp, qual reduz mais o indice total?
A resposta: Supply Chain (mas so一点点 — todas contribuem proporcionalmente ao peso).


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
dims_s = ['Cambio', 'Supply Chain', 'Regulatorio', 'Competitivo']
score_keys = ['cambio', 'supply', 'reg', 'comp']
weights_list = [30, 30, 20, 20]
scores_list = [scores[k] for k in score_keys]
marginal_impact = [w * s / 100 for w, s in zip(weights_list, scores_list)]
colors_sens = [C_RED if s > 70 else C_YELLOW if s > 50 else C_GREEN for s in scores_list]
bars = ax.barh(dims_s, marginal_impact, color=colors_sens, alpha=0.8, height=0.5)
for bar, w, s, mi in zip(bars, weights_list, scores_list, marginal_impact):
    ax.text(mi + 0.3, bar.get_y() + bar.get_height()/2,
            f'{w}%x{s:.0f} = {mi:.1f}pp', va='center', fontsize=9,
            fontweight='bold', color=bar.get_facecolor())
ax.axvline(total, color=C_YELLOW, lw=2, linestyle='--', label=f'Indice total: {total:.1f}/100')
ax.set_xlabel('Contribuicao para o Indice (pp)', fontsize=10)
ax.set_title('Contribuicao Marginal por Dimensao\n(Peso x Score)', fontweight='bold')
ax.legend(fontsize=9); ax.grid(True, alpha=0.2, axis='x')

ax = axes[1]
improvement = 10
current_scores = [scores[k] for k in score_keys]
indices_after = []
for dim_idx in range(4):
    new_scores = current_scores.copy()
    new_scores[dim_idx] = max(0, new_scores[dim_idx] - improvement)
    new_total = sum(weights_list[i] * new_scores[i] / 100 for i in range(4))
    indices_after.append(new_total)

reduction = [total - ia for ia in indices_after]
bars2 = ax.barh(dims_s, reduction, color=colors_sens, alpha=0.8, height=0.5)
for bar, red in zip(bars2, reduction):
    ax.text(red + 0.1, bar.get_y() + bar.get_height()/2,
            f'-{red:.1f}pp', va='center', fontsize=10, fontweight='bold',
            color=bar.get_facecolor())
ax.set_xlabel('Reducao do Indice Composto (pp)', fontsize=10)
ax.set_title(f'Sensibilidade: "Se eu melhorar {improvement}pp..."\n(Indice atual: {total:.1f})', fontweight='bold')
ax.grid(True, alpha=0.2, axis='x')
ax.text(0.98, 0.02,
        f'Reduzir Supply Chain de 95,7 para 85,7\n'
        f'diminui o indice em 3,0pp (de {total:.1f} para {total-3:.1f})',
        transform=ax.transAxes, ha='right', va='bottom', fontsize=8.5, color=C_RED, alpha=0.9,
        bbox=dict(boxstyle='round', facecolor='#161b22', edgecolor=C_RED, alpha=0.8))

fig.suptitle('CAPITULO 5B -- Sensibilidade: Onde Intervir para Maximo Impacto?',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('cap5b_sensitivity.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print('Figura salva: cap5b_sensitivity.png')


---
## CAPITULO 6 — Dashboard Final: O Painel Unico

Abaixo o dashboard completo que resume todos os capitulos em uma unica pagina visual.
Este e o "one-pager" que voce mostra para um gestor ou para si mesmo toda segunda-feira.


In [ ]:
def score_color(s):
    if s >= 80: return C_RED
    if s >= 60: return C_ORANGE
    if s >= 40: return C_YELLOW
    return C_GREEN

def score_label(s):
    if s >= 80: return 'CRITICO'
    if s >= 60: return 'ELEVADO'
    if s >= 40: return 'MODERADO'
    return 'BAIXO'

fig = plt.figure(figsize=(20, 14))
gs  = gridspec.GridSpec(3, 4, figure=fig, hspace=0.45, wspace=0.35)

# KW1: PTAX
ax = fig.add_subplot(gs[0, 0])
ax.text(0.5, 0.85, 'PTAX', fontsize=11, fontweight='bold', ha='center', transform=ax.transAxes)
ax.text(0.5, 0.65, f'BRL {D["ptax"]["atual"]}', fontsize=22, fontweight='bold',
        ha='center', color=C_BLUE, transform=ax.transAxes)
ax.text(0.5, 0.48, f'Vol: {D["ptax"]["vol30_annualized"]}% a.a.',
        fontsize=9, ha='center', color=C_GRAY, transform=ax.transAxes)
ax.text(0.5, 0.35, f'Trend 180d: {D["ptax"]["trend180d_pct"]:+.2f}%',
        fontsize=9, ha='center', color=C_GRAY, transform=ax.transAxes)
ax.text(0.5, 0.18, f'Cambio Score: {scores["cambio"]:.0f}/100',
        fontsize=10, fontweight='bold', ha='center', color=score_color(scores['cambio']), transform=ax.transAxes)
ax.text(0.5, 0.07, score_label(scores['cambio']), fontsize=8, ha='center',
        color=score_color(scores['cambio']), transform=ax.transAxes, alpha=0.8)
ax.set_title('1. Cambio', fontweight='bold', fontsize=11, color=C_BLUE)
ax.axis('off')

# KW2: Supply HHI
ax = fig.add_subplot(gs[0, 1])
ax.text(0.5, 0.85, 'Supply HHI', fontsize=11, fontweight='bold', ha='center', transform=ax.transAxes)
ax.text(0.5, 0.65, f'{D["supply_hhi"]["battery_cells"]}', fontsize=22, fontweight='bold',
        ha='center', color=C_RED, transform=ax.transAxes)
ax.text(0.5, 0.48, 'Baterias (CATL 65%)', fontsize=9, ha='center', color=C_GRAY, transform=ax.transAxes)
ax.text(0.5, 0.35, f'Litio: {D["supply_hhi"]["lithium"]} | Semi: {D["supply_hhi"]["semiconductors"]}',
        fontsize=8, ha='center', color=C_GRAY, transform=ax.transAxes)
ax.text(0.5, 0.18, f'Supply Score: {scores["supply"]:.0f}/100',
        fontsize=10, fontweight='bold', ha='center', color=score_color(scores['supply']), transform=ax.transAxes)
ax.text(0.5, 0.07, score_label(scores['supply']), fontsize=8, ha='center',
        color=score_color(scores['supply']), transform=ax.transAxes, alpha=0.8)
ax.set_title('2. Supply Chain', fontweight='bold', fontsize=11, color=C_RED)
ax.axis('off')

# KW3: Regulatorio
ax = fig.add_subplot(gs[0, 2])
ax.text(0.5, 0.85, 'Regulatorio', fontsize=11, fontweight='bold', ha='center', transform=ax.transAxes)
ax.text(0.5, 0.65, f'{D["reg_scenarios"]["continuidade"]:.0f}%', fontsize=22, fontweight='bold',
        ha='center', color=C_BLUE, transform=ax.transAxes)
ax.text(0.5, 0.48, 'Incentivos (Rota 2030)', fontsize=9, ha='center', color=C_GRAY, transform=ax.transAxes)
ax.text(0.5, 0.35, 'Rollback total: 0% | Expansao: 25%',
        fontsize=8, ha='center', color=C_GRAY, transform=ax.transAxes)
ax.text(0.5, 0.18, f'Reg Score: {scores["reg"]:.0f}/100',
        fontsize=10, fontweight='bold', ha='center', color=score_color(scores['reg']), transform=ax.transAxes)
ax.text(0.5, 0.07, score_label(scores['reg']), fontsize=8, ha='center',
        color=score_color(scores['reg']), transform=ax.transAxes, alpha=0.8)
ax.set_title('3. Regulatorio', fontweight='bold', fontsize=11, color=C_YELLOW)
ax.axis('off')

# KW4: Competitivo
ax = fig.add_subplot(gs[0, 3])
ax.text(0.5, 0.85, 'Competitivo', fontsize=11, fontweight='bold', ha='center', transform=ax.transAxes)
ax.text(0.5, 0.65, f'BYD {D["competition"]["byd"][0]*100:.0f}%', fontsize=22, fontweight='bold',
        ha='center', color=C_BLUE, transform=ax.transAxes)
ax.text(0.5, 0.48, f'2026 share lider', fontsize=9, ha='center', color=C_GRAY, transform=ax.transAxes)
ax.text(0.5, 0.35, f'Tesla: 5%->18% (ameaca 2027)',
        fontsize=8, ha='center', color=C_RED, transform=ax.transAxes)
ax.text(0.5, 0.18, f'Comp Score: {scores["comp"]:.0f}/100',
        fontsize=10, fontweight='bold', ha='center', color=score_color(scores['comp']), transform=ax.transAxes)
ax.text(0.5, 0.07, score_label(scores['comp']), fontsize=8, ha='center',
        color=score_color(scores['comp']), transform=ax.transAxes, alpha=0.8)
ax.set_title('4. Competitivo', fontweight='bold', fontsize=11, color=C_GREEN)
ax.axis('off')

# KW5: Composite Gauge
ax = fig.add_subplot(gs[1, :2])
score = total
theta_gauge = np.linspace(np.pi, 0, 100)
r_gauge = 1
for th in np.linspace(np.pi, 0, 100):
    color_g = C_GREEN if th > 2*np.pi/3 else C_YELLOW if th > np.pi/3 else C_RED
    ax.plot([0, r_gauge * np.cos(th)], [0, r_gauge * np.sin(th)],
            color=color_g, lw=18, alpha=0.4)

needle_angle = np.pi * (1 - score/100)
ax.arrow(0, 0, 0.7 * np.cos(needle_angle), 0.7 * np.sin(needle_angle),
         head_width=0.08, head_length=0.05, fc='white', ec='white', lw=2)
ax.text(0, -0.25, f'{score:.1f}/100', fontsize=28, fontweight='bold',
        ha='center', va='center', color=score_color(score))
ax.text(0, -0.45, 'INDICE COMPOSTO', fontsize=10, ha='center', color=C_GRAY, style='italic')
ax.text(0, -0.58, score_label(score), fontsize=12, fontweight='bold', ha='center', color=score_color(score))
ax.set_xlim(-1.3, 1.3); ax.set_ylim(-0.7, 1.1); ax.axis('off')
ax.set_title('5. Indice Composto -- Vulnerabilidade Geral', fontweight='bold', fontsize=12, pad=10)

# KW6: Monte Carlo
ax = fig.add_subplot(gs[1, 2:])
bom_sample = np.array(D['bom_impacts_sample'])
p5_s, p50_s, p95_s = np.percentile(bom_sample, 5), np.percentile(bom_sample, 50), np.percentile(bom_sample, 95)
prob_loss_s = (bom_sample < 0).mean() * 100
ax.text(0.5, 0.92, 'Monte Carlo -- Impacto no BOM (10k x 6m)',
        fontsize=10, fontweight='bold', ha='center', transform=ax.transAxes)
kpi_data = [
    ('P5 (Risco)', f'{p5_s:.2f}pp', C_RED),
    ('P50 (Base)', f'{p50_s:.2f}pp', C_YELLOW),
    ('P95 (Oportun.)', f'{p95_s:.2f}pp', C_GREEN),
    ('Prob(perda)', f'{prob_loss_s:.1f}%', C_RED if prob_loss_s > 40 else C_YELLOW),
]
for i, (lbl, val, clr) in enumerate(kpi_data):
    x = 0.15 + i * 0.22
    ax.text(x, 0.65, lbl, fontsize=8, ha='center', transform=ax.transAxes, color=C_GRAY)
    ax.text(x, 0.42, val, fontsize=16, fontweight='bold', ha='center', transform=ax.transAxes, color=clr)
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis('off')
ax.set_title('6. Simulacao Probabilistica', fontweight='bold', fontsize=12, pad=10)

# KW7: Timeline
ax = fig.add_subplot(gs[2, :])
risk_timeline = [
    {'t': '2026-Q3', 'event': 'Tesla Model 2\npre-venda Brasil', 'risk': 'COMP', 'level': 'MEDIUM'},
    {'t': '2026-Q4', 'event': 'Decisao Routa 2030\nrenovacao/rollback', 'risk': 'REG', 'level': 'HIGH'},
    {'t': '2027-Q1', 'event': 'Eleicoes presidenciais\n(impacto cambio)', 'risk': 'CAMBIO', 'level': 'HIGH'},
    {'t': '2027-Q2', 'event': 'Lancamento\nTesla Model 2', 'risk': 'COMP', 'level': 'CRITICAL'},
    {'t': '2027-Q3', 'event': 'Greve nacional\n(risco supply)', 'risk': 'SUPPLY', 'level': 'MEDIUM'},
    {'t': '2028-Q1', 'event': 'Expansao VW/GM\nline-up EV', 'risk': 'COMP', 'level': 'HIGH'},
]
level_colors = {'LOW': C_GREEN, 'MEDIUM': C_YELLOW, 'HIGH': C_ORANGE, 'CRITICAL': C_RED}
for i, ev in enumerate(risk_timeline):
    x = i / (len(risk_timeline) - 1)
    clr = level_colors[ev['level']]
    ax.scatter(x, 0.5, s=400, color=clr, alpha=0.8, zorder=5, edgecolors='white', linewidths=2)
    ax.text(x, 0.5, ev['t'].split('-')[1], ha='center', va='center',
            fontsize=7, fontweight='bold', color='white')
    ax.text(x, 0.22, ev['event'], ha='center', va='top', fontsize=7.5,
            color=C_GRAY, multialignment='center')
    ax.text(x, 0.78, ev['risk'], ha='center', va='bottom', fontsize=8,
            fontweight='bold', color=clr)
    ax.text(x, 0.88, ev['level'], ha='center', va='bottom', fontsize=7, color=clr, alpha=0.8)
ax.axhline(0.5, color=C_GRAY, lw=1, alpha=0.3)
ax.set_xlim(-0.05, 1.05); ax.set_ylim(0, 1); ax.axis('off')
ax.set_title('Timeline de Riscos -- 2026 a 2028 (Eventos-Chave)', fontweight='bold', fontsize=12, pad=10)

fig.text(0.5, 0.01,
         f'BYD Brasil -- Dashboard de Vulnerabilidade  |  Atualizado: {D["computed_at"]}  |  '
         f'PTAX: BRL {D["ptax"]["atual"]}  |  Composite: {total:.1f}/100  |  '
         f'Produzido por: Matheus Mendes',
         ha='center', fontsize=8, color=C_GRAY, style='italic')
fig.suptitle('DASHBOARD COMPLETO -- BYD Brasil / Camacari 2025-2027',
             fontsize=16, fontweight='bold', y=0.98)
plt.savefig('cap6_dashboard_final.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print('Figura salva: cap6_dashboard_final.png')


---
## CAPITULO 7 — Scorecard Final

Abaixo o scorecard numerico consolidado. Este e o resumo que voce usa para acompanhamento
trimestral — comparar os scores de hoje com os de ontem.


In [ ]:
print('=' * 70)
print('RESUMO EXECUTIVO -- BYD BRASIL / CAMACARI 2025-2027')
print('=' * 70)
print()
print(f"Data de referencia : {D['computed_at']}")
print(f"PTAX atual        : BRL {D['ptax']['atual']} (17/jul/2026)")
print(f"Volatilidade 30d  : {D['ptax']['vol30_annualized']}% a.a.")
print(f"Tendencia 180d    : {D['ptax']['trend180d_pct']:+.2f}% (depreciacao)")
print()
print('-' * 70)
print('INDICE COMPOSTO DE VULNERABILIDADE')
print('-' * 70)
for dim_key in ['cambio', 'supply', 'reg', 'comp']:
    label = {'cambio': 'Cambio', 'supply': 'Supply Chain', 'reg': 'Regulatorio', 'comp': 'Competitivo'}[dim_key]
    print(f"  {label:15s} : {scores[dim_key]:5.1f}/100  [{score_label(scores[dim_key])}]")
print(f"  {'-'*15} : {'-'*10}")
print(f"  {'COMPOSTO TOTAL':15s} : {total:5.1f}/100  [{score_label(total)}]")
print()
print('-' * 70)
print('MONTE CARLO -- 10.000 SIMULACOES x 6 MESES')
print('-' * 70)
print(f"  P5  (risco severo, 90% conf)  : {p5:.2f}pp")
print(f"  P50 (mediana/base)            : {p50:.2f}pp")
print(f"  P95 (otimismo)                : {p95:.2f}pp")
print(f"  Probabilidade de perda        : {prob_loss:.1f}%")
print()
print('-' * 70)
print('STRESS TEST -- APRECIACAO DO REAL')
print('-' * 70)
for s in D['stress']:
    print(f"  {s:>5} PTAX BRL {D['stress'][s]['ptax']:.4f}  ->  BOM {D['stress'][s]['bom']:+.1f}pp")
print()
print('-' * 70)
print('SUPPLY CHAIN -- HHI')
print('-' * 70)
print(f"  Celulas de Bateria : HHI {D['supply_hhi']['battery_cells']} (CATL 65%)")
print(f"  Litio              : HHI {D['supply_hhi']['lithium']}")
print(f"  Semicondutores     : HHI {D['supply_hhi']['semiconductors']} (TSMC 40%)")
print()
print('-' * 70)
print('MARKET SHARE -- PROJECAO 2026-2028')
print('-' * 70)
for brand in ['byd', 'tesla', 'vw', 'gm', 'others']:
    vals = comp[brand]
    delta = (vals[-1] - vals[0]) * 100
    print(f"  {brand.upper():8s}  {vals[0]*100:5.1f}% (2026) -> {vals[-1]*100:5.1f}% (2028)  d={delta:+.1f}pp")
print()
print('=' * 70)
print('PRIORIDADES DE ACAO')
print('=' * 70)
print("  1. [CRITICA]   Qualificar fornecedor #2 de baterias (Samsung SDI ou LG)")
print("  2. [ALTA]      Implementar hedge cambial NDF 6m (50% da exposicao)")
print("  3. [ALTA]      Engajar lobbying Rota 2030 via ABVE/ANFAVEA")
print("  4. [MEDIA]     Monitorar lancamento Tesla Model 2 (Q2 2027)")
print("  5. [CONTINUA]  Refresh trimestral dos dados (scripts prontos)")
print('=' * 70)
